# Your First Agent

**Course:** Build Production AI Agents with the Claude Agent SDK

---
## Step 1 — One‑Time Setup

Install the SDK once and load your API key once. Every later cell in this notebook reuses this same environment — no repeated `%pip install`, no repeated `load_dotenv()`.

In [ ]:
# We pin the Claude Agent SDK to a specific version so every example in this
# notebook runs exactly as shown in the course. Remove the version pin
# ('claude-agent-sdk==0.2.93' -> 'claude-agent-sdk') if you want the latest release.

%pip install claude-agent-sdk==0.2.93 python-dotenv -q

from dotenv import load_dotenv
import os

load_dotenv()

print("SDK installed successfully!")
print("API key set:", os.environ.get("ANTHROPIC_API_KEY") is not None)

### Model configuration

One variable controls which Claude model every agent call in this notebook uses. Change it here once instead of in every cell.

In [ ]:
# Change this to use a different Claude model
# e.g. "claude-sonnet-4-5", "claude-opus-4-5"
MODEL_NAME = "claude-haiku-4-5"

---
## Step 2 — Writing Our First `query()` Call

The `query()` function is the heart of the Claude Agent SDK.

You give it:
- A **`prompt`** — what you want the agent to do
- A **`ClaudeAgentOptions`** object — configuration, including which tools the agent is allowed to use

It returns an **async stream of messages**. You loop over that stream with `async for`. We print every message raw first, so you see exactly what comes back before we decode it in Step 3.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions

async def explore_files():
    async for message in query(
        prompt="What files are in this directory?",
        options=ClaudeAgentOptions(allowed_tools=["Bash", "Glob"], model=MODEL_NAME),
    ):
        print(message)

await explore_files()

### What just happened?

1. Claude received the prompt and the list of allowed tools, and entered its internal agent loop.
2. Claude chose a tool (`Bash` or `Glob`) on its own — you never specified which.
3. The tool ran and returned a result; Claude read it and decided whether it had enough information.
4. Claude exited the loop and returned a final answer as the last message — a `ResultMessage`.
5. You looped over the whole stream with `async for`.

That raw output is dense. Let's decode it.

```mermaid

sequenceDiagram
    autonumber

    box rgb(230,240,255) Your Code [App]
        participant APP as Python App
    end

    box rgb(235,250,240) Claude Agent SDK
        participant SDK as Claude Agent SDK
    end

    box rgb(255,245,225) Claude LLM
        participant LLM as Claude LLM
    end

    box rgb(245,235,255) Claude Code [Tool Calling]
        participant CODE as Claude Code
    end

    APP->>SDK: query(prompt, ClaudeAgentOptions)
    Note over APP,SDK: allowed_tools = ["Glob", "Read"]<br/>model = MODEL_NAME

    SDK->>LLM: Send user prompt + available tools

    Note over LLM: Understand task:<br/>Explore sample_project<br/>List files → Read files → Summarize → Report

    LLM-->>SDK: Tool request: Glob<br/>"Find all files and folders"

    SDK->>CODE: Execute Glob
    CODE->>CODE: Search sample_project
    CODE-->>SDK: File/folder paths

    SDK-->>LLM: Glob results

    Note over LLM: Analyze discovered files<br/>Determine which files to read

    loop For each discovered file
        LLM-->>SDK: Tool request: Read(file)

        SDK->>CODE: Execute Read(file)
        CODE->>CODE: Open and read file
        CODE-->>SDK: File contents

        SDK-->>LLM: File contents

        Note over LLM: Understand file<br/>Create one-sentence summary
    end

    Note over LLM: Compile all summaries<br/>Create clean readable report

    LLM-->>SDK: Final result / report

    SDK-->>APP: message.result

    APP->>APP: print(message.result)

```


---
## Step 3 — Reading Agent Messages & Content Blocks

**The foundational insight:** the SDK is the only thing that writes to your stream — always. Claude returns JSON over HTTP; the SDK constructs every Python object you see. The distinction between message types is about **who decided the content**, not who surfaced it.

### The four top-level message types

| Message Type | Generated by | When | Key fields |
|---|---|---|---|
| `SystemMessage` | **SDK** | Before Claude sees anything | `subtype`, `data` (has `session_id`) |
| `AssistantMessage` | **Claude** (packaged by SDK) | Every Claude response | `content` (list of blocks) |
| `UserMessage` | **SDK** | After each tool executes | `content` (list of blocks) |
| `ResultMessage` | **SDK** | Once, loop complete | `result`, `total_cost_usd`, `num_turns`, `duration_ms` |

> There is **no** `ToolUseMessage` or `ToolResultMessage`. Tool activity lives inside `AssistantMessage` and `UserMessage` as **content blocks**.

### Content blocks

**Inside `AssistantMessage.content`:**
| Block | Key fields | Meaning |
|---|---|---|
| `TextBlock` | `block.text` | Claude's plain text output |
| `ToolUseBlock` | `block.name`, `block.input` | Claude calling a tool |
| `ThinkingBlock` | `block.thinking` | Claude's extended reasoning |

**Inside `UserMessage.content`:**
| Block | Key fields | Meaning |
|---|---|---|
| `ToolResultBlock` | `block.content` | Result returned from a tool execution |

**The two-level pattern:**
```
Check isinstance on the outer message
  └── if AssistantMessage or UserMessage:
        loop through message.content
        └── check isinstance on each block
```

Below we run the **same call from Step 2** again, this time labelling every message and block as it arrives — no new example, just the first one decoded.

In [ ]:
from claude_agent_sdk import (
    SystemMessage,
    AssistantMessage,
    UserMessage,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    ToolResultBlock,
    ThinkingBlock,
)


async def explore_files_labelled():
    async for message in query(
        prompt="What files are in this directory?",
        options=ClaudeAgentOptions(allowed_tools=["Bash", "Glob"], model=MODEL_NAME),
    ):
        if isinstance(message, SystemMessage):
            # Generated by the SDK -- before Claude sees anything
            print(f"[SYSTEM]       SDK init -- subtype={message.subtype}")

        elif isinstance(message, AssistantMessage):
            # Content decided by Claude, packaged by SDK
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"[TEXT]         {block.text[:80]}")
                elif isinstance(block, ToolUseBlock):
                    print(f"[TOOL USE]     tool={block.name}  input={block.input}")
                elif isinstance(block, ThinkingBlock):
                    print(f"[THINKING]     {block.thinking[:60]}...")

        elif isinstance(message, UserMessage):
            # Generated by the SDK -- tool result fed back to Claude
            for block in message.content:
                if isinstance(block, ToolResultBlock):
                    print(f"[TOOL RESULT]  {str(block.content)[:80]}")

        elif isinstance(message, ResultMessage):
            # Generated by the SDK -- loop complete, once only
            print(f"[RESULT]       {message.result[:80]}")
            print(f"               cost=${message.total_cost_usd:.4f}  "
                  f"turns={message.num_turns}  "
                  f"duration={message.duration_ms}ms")


await explore_files_labelled()

# Pattern: two levels
#   1. isinstance on the outer message
#   2. loop through message.content -> isinstance on each block
# [:80] just keeps notebook output tidy -- in production read full content

---
## Step 4 — The Clean Production Pattern

In most real agents — a CI/CD pipeline, a backend service, a user-facing app — you only need the **final answer**. This is the pattern you will use throughout the rest of the course.

### Why `hasattr(message, "result")` and not `TextBlock`?

`AssistantMessage` with `TextBlock` can appear **multiple times** during a run, including as intermediate commentary between tool calls.

`ResultMessage` appears **exactly once**, at the very end. Its `.result` field contains the same text as the last `AssistantMessage` `TextBlock` — the SDK copies it there — but it also gives you `total_cost_usd`, `num_turns`, `duration_ms`, and `is_error`.

`hasattr(message, "result")` waits for the SDK's signal that the task is **fully complete**.

In [ ]:
async def explore_files_clean():
    async for message in query(
        prompt="What files are in this directory?",
        options=ClaudeAgentOptions(allowed_tools=["Bash", "Glob"], model=MODEL_NAME),
    ):
        if hasattr(message, "result"):
            print(message.result)


await explore_files_clean()

# WHY hasattr?
#   Only ResultMessage has a .result field.
#   It fires exactly once -- when the SDK signals the loop is fully complete.
#   The text is identical to the last AssistantMessage TextBlock, plus cost/turn/duration metadata.
#
# ALTERNATIVE (equally correct, more explicit):
#   if isinstance(message, ResultMessage):
#       print(message.result)
#
# DEFAULT throughout this course: hasattr(message, "result")

---
## Step 5 — When Would You Read Content Blocks Instead?

`hasattr(message, "result")` is the right default. But here are three real-world scenarios where reading content blocks (Step 3's pattern) is the correct approach:

### 🔍 Debugging
Loop through `AssistantMessage.content` and read `ToolUseBlock` to see exactly what tool was called and what input was passed. Read `ToolResultBlock` from `UserMessage` to see what came back.

```python
elif isinstance(message, AssistantMessage):
    for block in message.content:
        if isinstance(block, ToolUseBlock):
            print(f"Tool called: {block.name}")
            print(f"Input:       {block.input}")

elif isinstance(message, UserMessage):
    for block in message.content:
        if isinstance(block, ToolResultBlock):
            print(f"Result:      {block.content}")
```

### 📺 Real-Time UI Progress
As `ToolUseBlock` objects arrive inside `AssistantMessage`, update a progress indicator in your frontend — *"Claude is searching..."*, *"Running tests..."*

### 📋 Audit Logging (→ Section 5: Hooks)
Log every `ToolUseBlock` to a file for compliance or auditing.

**Summary:** default to `hasattr(message, "result")`. Read content blocks when you need visibility, real-time feedback, or audit trails.

---

### Recap so far

- The SDK is the only thing that writes to your stream — Claude decides content, the SDK packages it.
- Four message types: `SystemMessage`, `AssistantMessage`, `UserMessage`, `ResultMessage`. No `ToolUseMessage`/`ToolResultMessage` — tool activity is always a block inside a message.
- Two-level pattern: `isinstance` on the message → loop `message.content` → `isinstance` on the block.
- Production default: `hasattr(message, "result")`.
- Your prompt is never echoed back — the first message in the stream is always `SystemMessage`.

Now let's put all of this together in one complete, realistic agent.

---
## Step 6 — Hands-On: Build a File Explorer Agent

An agent that autonomously explores a project directory, reads every file, and produces a clean summary report. You describe the goal in plain English — the agent handles the rest.

### The Colab / local filesystem

- Every session can start with a clean slate, so we create sample files **programmatically** — full control, reproducibility, and predictable output to check the agent's summary against.
- `Read`, `Glob`, `Grep` all operate on this filesystem exactly as they would on a real machine.

In [ ]:
# Create the sample project structure and files
import os

os.makedirs("sample_project/src", exist_ok=True)
os.makedirs("sample_project/tests", exist_ok=True)

with open("sample_project/README.md", "w") as f:
    f.write("# Sample Project\n")
    f.write("This is a demo project for the File Explorer Agent.\n")
    f.write("It contains an authentication module, utility functions, and tests.\n")

with open("sample_project/src/auth.py", "w") as f:
    f.write("# Authentication module\n\n")
    f.write("def login(user, password):\n")
    f.write('    """Authenticate a user with username and password."""\n')
    f.write("    return user == 'admin' and password == 'secret'\n\n")
    f.write("def logout(user):\n")
    f.write('    """Log out the specified user."""\n')
    f.write("    return True\n\n")
    f.write("def reset_password(user, new_password):\n")
    f.write('    """Reset the password for the specified user."""\n')
    f.write("    return True\n")

with open("sample_project/src/utils.py", "w") as f:
    f.write("# Utility functions\n\n")
    f.write("def format_date(date):\n")
    f.write('    """Format a date object as YYYY-MM-DD string."""\n')
    f.write("    return date.strftime('%Y-%m-%d')\n\n")
    f.write("def format_currency(amount):\n")
    f.write('    """Format a number as a currency string."""\n')
    f.write("    return f'${amount:.2f}'\n\n")
    f.write("def validate_email(email):\n")
    f.write('    """Check if an email address is valid."""\n')
    f.write("    return '@' in email\n")

with open("sample_project/tests/test_auth.py", "w") as f:
    f.write("# Tests for auth module\n\n")
    f.write("def test_login():\n")
    f.write('    """Test the login function."""\n')
    f.write("    assert login('admin', 'secret') == True\n\n")
    f.write("def test_logout():\n")
    f.write('    """Test the logout function."""\n')
    f.write("    assert logout('admin') == True\n")

print("Sample project created successfully.")

### Verify what was created

Print the directory structure and every file's contents before the agent runs, so we know what its summary should say — no black boxes.

In [ ]:
print("Project structure:")
print("=" * 40)
for root, dirs, files in os.walk("sample_project"):
    level = root.replace("sample_project", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

files_to_show = [
    "sample_project/README.md",
    "sample_project/src/auth.py",
    "sample_project/src/utils.py",
    "sample_project/tests/test_auth.py",
]

print("\nFile contents:")
print("=" * 40)
for filepath in files_to_show:
    print(f"\n--- {filepath} ---")
    with open(filepath, "r") as f:
        print(f.read())

### Build the agent

**Two tools:**
- **`Glob`** — finds files by pattern, like a powerful `find` command (e.g. `**/*.py`)
- **`Read`** — opens and reads any file in the working directory

We give the agent a goal in plain English using a numbered prompt — it figures out the tool sequence itself. We extract the final result using the **Step 4 production pattern**: `hasattr(message, "result")`.

In [ ]:
async def file_explorer_agent():
    async for message in query(
        prompt="""
        Explore the project at sample_project.
        1. List all files and folders in the project
        2. Read each file
        3. For each file, write a one sentence summary of what it contains
        4. Present the results as a clean, readable report
        """,
        options=ClaudeAgentOptions(
            allowed_tools=["Glob", "Read"],
            model=MODEL_NAME,
        ),
    ):
        if hasattr(message, "result"):
            print(message.result)


await file_explorer_agent()

### What just happened?

- You did not tell it to use `Glob` before `Read`.
- You did not write a loop to open each file.
- You described a **goal** — the agent figured out the tool sequence itself.

That is the agent loop running live on a real task: receive the goal → decide which tool to call → call it, get the result → decide the next step → repeat until the goal is met → return the final answer as a `ResultMessage`.

---
## Step 7 — The Prompt Is Your Primary Control Lever

**Change the prompt, change the agent** — without touching a single line of code. Same tools (`Glob` + `Read`), same `ClaudeAgentOptions`, same `hasattr(message, "result")` pattern — only the prompt changes, from "summarize each file" to "list every function."

In [ ]:
async def function_inventory_agent():
    async for message in query(
        prompt="""
        Explore the project at sample_project.
        1. Find all Python files only
        2. For each Python file, list all the function names defined in it
        3. Present the results as a clean report
        """,
        options=ClaudeAgentOptions(
            allowed_tools=["Glob", "Read"],
            model=MODEL_NAME,
        ),
    ):
        if hasattr(message, "result"):
            print(message.result)


await function_inventory_agent()

---
## Summary — The Whole Flow, End to End

| Concept | What it is |
|---------|------------|
| One-time setup | Install the SDK once, load the API key once — reused for every call in this notebook |
| `MODEL_NAME` | One variable controlling which Claude model every agent call uses |
| `query()` + `ClaudeAgentOptions` | The core call: a prompt, a set of allowed tools, an async stream of messages |
| Four message types | `SystemMessage`, `AssistantMessage`, `UserMessage`, `ResultMessage` — no separate tool-use/tool-result message types |
| Two-level pattern | `isinstance` on the message → loop `message.content` → `isinstance` on the block |
| `hasattr(message, "result")` | The default production pattern — fires once, when the run is fully complete |
| Content blocks | Read them for debugging, real-time UI progress, or audit logging — not by default |
| Verification pattern | Print the project before the agent runs, so you can check its summary against ground truth |
| `Glob` + `Read` | Two tools that together give complete filesystem read access |
| Prompt as control lever | Same code, same tools — a different prompt produces a completely different agent behaviour |

---

**Coming up next:** Lecture 2.5 — Troubleshooting Common Setup Errors.

*Great work — you now have the entire Section 2 flow in one place, built up step by step with nothing repeated.*